#**Plano de Ação: Detecção de Fraudes com XGBoost**

Nosso objetivo é construir um modelo de Machine Learning que consiga distinguir com alta precisão transações fraudulentas de transações legítimas. Seguiremos um pipeline bem definido, desde a análise dos dados até a submissão final.

#**Passo 0: Configuração do Ambiente e Bibliotecas**

Antes de tudo, precisamos garantir que nosso ambiente de desenvolvimento (seja um notebook local, Google Colab, etc.) tenha as ferramentas necessárias.

In [ ]:
# Instalação das bibliotecas essenciais
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn

#**Passo 1: Análise Exploratória de Dados (EDA - Exploratory Data Analysis)**

Esta é a fase mais crucial. Precisamos "conversar" com os dados para entender suas características, padrões e desafios.

**1. Carregar os Dados:** Vamos carregar os dois datasets fornecidos.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Carregando os datasets
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

print("Dimensões do treino:", df_train.shape)
print("Dimensões do teste:", df_test.shape)

**2. Inspeção Inicial:** Olhar as primeiras linhas, os tipos de dados e estatísticas descritivas.

In [ ]:
print(df_train.head())
print(df_train.info())
print(df_train.describe())

**3. Verificar o Desbalanceamento de Classes:** Este é o principal desafio em detecção de fraudes. Haverá muito mais transações legítimas do que fraudulentas.

In [ ]:
# Assumindo que a coluna alvo se chama 'isFraud' (ajuste se for outro nome)
target_column = 'Class'
print(df_train[target_column].value_counts(normalize=True))

# Visualizando o desbalanceamento
sns.countplot(x=target_column, data=df_train)
plt.title('Distribuição das Classes (0 = Legítima, 1 = Fraude)')
plt.show()

**4. Análise de Valores Ausentes (Missing Values):** Verificar quais colunas têm dados faltando e decidir como tratá-los.

In [ ]:
print(df_train.isnull().sum().sort_values(ascending=False))

**5. Análise de Correlação e Distribuição:** Entender a relação entre as features e a variável alvo.

In [ ]:
# Heatmap de correlação (para features numéricas)
plt.figure(figsize=(12, 10))
sns.heatmap(df_train.corr(), annot=False, cmap='coolwarm')
plt.title('Mapa de Correlação')
plt.show()

#**Passo 2: Pré-processamento e Engenharia de Features (Feature Engineering)**

Aqui transformamos os dados brutos em um formato ideal para o XGBoost e criamos novas features que possam dar mais "poder" ao modelo.

**1. Tratamento de Valores Ausentes:**

a) Para features numéricas, podemos preencher com a média, mediana ou um valor constante (como -999). A mediana é geralmente mais robusta a outliers.

b) Para features categóricas, podemos preencher com a moda (valor mais frequente).

**2. Engenharia de Features (Onde a mágica acontece):**

**a) Features de Tempo:** Se houver colunas de timestamp, podemos extrair informações valiosas: hora do dia, dia da semana, mês. Fraudes costumam ocorrer em horários específicos (ex: madrugada).

**b) Features de Agregação:** Criar novas features baseadas em agrupamentos. Exemplo: valor_medio_transacao_por_usuario, frequencia_transacoes_por_cartao.

**c) Features de Interação:** Combinar duas ou mais features. Exemplo: valor_transacao / saldo_conta_antes.

**3. Codificação de Variáveis Categóricas:**

O XGBoost lida bem com números. Se houver colunas de texto (ex: tipo_cartao, pais), precisamos convertê-las. A técnica mais comum é o **One-Hot Encoding**.

# **Passo 3: Modelagem com XGBoost**

**1. Definir Features (X) e Alvo (y):**

In [18]:
# Alinhar colunas do treino e teste para garantir consistência
train_labels = df_train[target_column]
train_ids = df_train['id'] # Ajuste o nome da coluna de ID
test_ids = df_test['id'] # Ajuste o nome da coluna de ID

# Remover colunas que não são features
df_train = df_train.drop(columns=[target_column, 'id'])
df_test = df_test.drop(columns=['id'])

# Garantir que ambos os dataframes tenham as mesmas colunas
common_cols = list(set(df_train.columns) & set(df_test.columns))
df_train = df_train[common_cols]
df_test = df_test[common_cols]

X = df_train
y = train_labels
X_test = df_test

**2. Configurar o XGBoost Classifier:**



*   A chave para lidar com o desbalanceamento é o parâmetro 'scale_pos_weight'. Ele informa ao modelo para dar mais importância aos erros na classe minoritária (fraudes).



In [19]:
# Calcular o scale_pos_weight
ratio = float(np.sum(y == 0)) / np.sum(y == 1)

# Instanciar o modelo com parâmetros iniciais robustos
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',  # Objetivo de classificação binária
    eval_metric='auc',            # Métrica de avaliação para a competição
    n_estimators=1000,            # Número de árvores (ajustaremos com early stopping)
    learning_rate=0.05,           # Taxa de aprendizado
    max_depth=5,                  # Profundidade máxima das árvores
    subsample=0.8,                # Fração de amostras usadas por árvore
    colsample_bytree=0.8,         # Fração de features usadas por árvore
    gamma=0.1,                    # Parâmetro de regularização
    scale_pos_weight=ratio,       # !! Parâmetro crucial para dados desbalanceados !!
    use_label_encoder=False,      # Evitar warnings
    n_jobs=-1,                    # Usar todos os cores do CPU
    random_state=42
)

**3. Treinar o Modelo com Validação e Early Stopping:**



*   Vamos separar uma parte dos dados de treino para validação. Isso nos permite monitorar o desempenho do modelo e parar o treinamento quando ele começar a piorar (overfitting), encontrando o número ideal de 'n_estimators'.



In [ ]:
# Criar um conjunto de validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Treinar o modelo
xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50, # Removido: Não suportado por esta versão do XGBoost
    # verbose=100 # Removido: Não suportado por esta versão do XGBoost
)

#**Passo 4: Avaliação do Modelo**

Após o treino, vamos verificar a performance no conjunto de validação que separamos.

In [ ]:
# Fazer previsões no conjunto de validação
preds_val = xgb_clf.predict_proba(X_val)[:, 1]

# Calcular o AUC-ROC
auc_score = roc_auc_score(y_val, preds_val)
print(f"AUC no conjunto de validação: {auc_score:.5f}")

# Analisar a Matriz de Confusão
preds_binary = (preds_val > 0.5).astype(int)
print("\nMatriz de Confusão:")
print(confusion_matrix(y_val, preds_binary))
print("\nRelatório de Classificação:")
print(classification_report(y_val, preds_binary))

#**Passo 5: Geração do Arquivo de Submissão para o Kaggle**

Se estivermos satisfeitos com a performance na validação, treinamos o modelo com todos os dados de treino e fazemos as previsões no conjunto de teste ('test.csv').

In [ ]:
# Treinar o modelo final com todos os dados de treino
# (Usando o número de árvores encontrado pelo early stopping, se desejado)
# xgb_clf.fit(X, y, ...) # Opcional: re-treinar com tudo

# Fazer previsões no conjunto de teste real
test_predictions = xgb_clf.predict_proba(X_test)[:, 1]

# Criar o arquivo de submissão no formato exigido pelo Kaggle
submission = pd.DataFrame({
    'TransactionID': test_ids,
    target_column: test_predictions
})

# Salvar o arquivo
submission.to_csv('submission.csv', index=False)

print("\nArquivo 'submission.csv' gerado com sucesso!")